# Brazilian E-Commerce Analysis

This project analyzes Brazilian e-commerce marketplace performance using transactional, customer, logistics, and review data from Olist.

The analysis focuses on:
- revenue and sales performance,
- customer satisfaction and logistics operations,
- customer retention and segmentation,
- and actionable business recommendations supported by data analysis and visualization.

---

**Dataset:** Olist Brazilian E-Commerce (Kaggle)  
**Author:** Nataliia Butenko  
**Date:** May 2026  
**Tools and Technologies:** Python (Pandas, Matplotlib, Seaborn, SciPy), SQL, BigQuery, Tableau

---
<a id="project-goal"></a>
## Project Goal

The goal of this project is to analyze Brazilian e-commerce marketplace performance and identify key factors affecting revenue growth, customer satisfaction, delivery operations, and customer retention.

The project combines exploratory data analysis, statistical testing, customer segmentation, and BI visualization to generate actionable insights and business recommendations.

---
<a id="dataset-description"></a>
## Dataset Description

The dataset contains transactional data from Olist — one of the largest Brazilian e-commerce marketplaces.

It includes customer, seller, product, payment, delivery, and review information covering the period from September 2016 to August 2018.

The analysis is based on:
- 99,441 orders across 8 interconnected tables,
- customer, seller, product, payment, and review data,
- and geographic information across 27 Brazilian states.

---

## Table of Contents

- [Project Goal](#project-goal)
- [Dataset Description](#dataset-description)

### Data Preparation

- [Step I. Environment and Data Setup](#step-i-environment-and-data-setup)
- [Step II. Exploratory Data Analysis](#step-ii-exploratory-data-analysis)
- [Step III. Data Cleaning and Preparation](#step-iii-data-cleaning-and-preparation)
  - [1. Data Type Conversion](#1-data-type-conversion)
  - [2. Handling Missing Values](#2-handling-missing-values)
  - [3. Feature Engineering](#3-feature-engineering)
- [Step IV. Creating Master Table](#step-iv-creating-master-table)

### Business Analysis

- [Executive Business Overview](#executive-business-overview)
  - [Performance KPIs](#performance-kpis)

#### Revenue and Sales Performance

- [Revenue and Order Volume Over Time](#revenue-and-order-volume-over-time)
- [Revenue Contribution by Product Category](#revenue-contribution-by-product-category)
- [Revenue Contribution by State](#revenue-contribution-by-state)
- [Seller Performance and Customer Satisfaction](#seller-performance-and-customer-satisfaction)
- [Payment Method Analysis](#payment-method-analysis)

#### Customer Experience and Logistics Performance

- [Impact of Delivery Delays on Customer Satisfaction](#impact-of-delivery-delays-on-customer-satisfaction)
- [Hypothesis Test: Late Deliveries and Review Scores ](#hypothesis-test-late-deliveries)
- [Delivery Performance and Customer Satisfaction by State](#delivery-performance-and-customer-satisfaction-by-state)
- [Low-Rated Product Categories and Customer Dissatisfaction](#low-rated-product-categories-and-customer-dissatisfaction)

#### Customer Retention and Segmentation

- [Repeat Purchase Behavior](#repeat-purchase-behavior)
- [Customer Segmentation Analysis (RFM)](#customer-segmentation-analysis-rfm)
- [Cohort Retention Analysis](#cohort-retention-analysis)

### Final Section

- [Conclusions and Business Recommendations](#conclusions-and-business-recommendations)

- [Tableau Dashboard](#tableau-dashboard)

<a id="step-i-environment-and-data-setup"></a>
## Step I. Environment and Data Setup

The analysis was performed using Python libraries for data manipulation, statistical analysis, and data visualization.

Key libraries:
- Pandas and NumPy for data processing
- Matplotlib and Seaborn for visualization
- SciPy for statistical testing

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

The project uses the Brazilian E-Commerce Public Dataset by Olist, which contains information about orders, customers, products, sellers, payments, and customer reviews.

The datasets were loaded separately and later combined into a analytical master table for business analysis and dashboard creation.

In [ ]:
# Loading raw datasets 
orders       = pd.read_csv("data/olist_orders_dataset.csv")
customers    = pd.read_csv("data/olist_customers_dataset.csv")
order_items  = pd.read_csv("data/olist_order_items_dataset.csv")
products     = pd.read_csv("data/olist_products_dataset.csv")
sellers      = pd.read_csv("data/olist_sellers_dataset.csv")
payments     = pd.read_csv("data/olist_order_payments_dataset.csv")
reviews      = pd.read_csv("data/olist_order_reviews_dataset.csv")
categories   = pd.read_csv("data/product_category_name_translation.csv")

<a id="step-ii-exploratory-data-analysis"></a>
## Step II. Exploratory Data Analysis

Before cleaning and transforming the datasets, an initial exploratory analysis was performed to better understand the structure and quality of the raw data.

For each dataset, the following checks were conducted:

- dataset dimensions
- column data types
- missing values
- duplicate records
- sample observations

This step helps identify potential data quality issues and guides subsequent cleaning and feature engineering processes.

In [ ]:
# Exploratory Data Analysis 
def explore_df(df, name):
    """Print shape, dtypes, sample, null counts for any dataframe."""
    print(f"\n{'='*50}")
    print(f"DATAFRAME NAME: {name}")
    print(f"{'='*50}")
    print(f"\nSHAPE: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\nCOLUMN TYPES:\n{df.dtypes}")
    nulls = df.isnull().sum()
    print(f"\nMISSING VALUES:\n{nulls[nulls > 0] if nulls.any() else 'None'}")
    print(f"\nDUPLICATES: {df.duplicated().sum()}")
    display(df.head(3))

The exploration was applied consistently across all source tables, including orders, customers, products, sellers, payments, and customer reviews.

In [ ]:
for df, name in [
    (orders,      "orders"),
    (customers,   "customers"),
    (order_items, "order_items"),
    (products,    "products"),
    (sellers,     "sellers"),
    (payments,    "payments"),
    (reviews,     "reviews"),
    (categories,  "categories"),
]:
    explore_df(df, name)

<a id="step-iii-data-cleaning-and-preparation"></a>
## Step III. Data Cleaning and Preparation

After completing the exploratory data analysis (EDA), the next step was to prepare the datasets for analytical work.  
The objective of data cleaning was not only to remove inconsistencies but also to preserve meaningful business information.

<a id="1-data-type-conversion"></a>
### 1. Data Type Conversion
Several columns containing timestamps were initially stored as string values. These columns were converted to datetime format to enable time-based analysis.

Converted columns:

- order_purchase_timestamp
- order_approved_at
- order_delivered_carrier_date
- order_delivered_customer_date
- order_estimated_delivery_date
- review_creation_date
- shipping_limit_date
- review_creation_date
- review_answer_timestamp

In [ ]:
# Changing date an time columns type to datetime format
def parse_date_columns(df, date_cols):
    """Convert specified columns to datetime format."""
    df[date_cols] = df[date_cols].apply(pd.to_datetime, errors="coerce")
    return df


# DataFrame: orders
orders = parse_date_columns(orders, [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
])

# DataFrame: order_items
order_items = parse_date_columns(order_items, ["shipping_limit_date"])

# DataFrame: reviews
reviews = parse_date_columns(reviews, ["review_creation_date", "review_answer_timestamp"])

# Checking the result
date_cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

date_cols_review = ["review_creation_date", "review_answer_timestamp"]

print(f"{orders[date_cols_orders].dtypes}")
print(f"\nshipping_limit_date    {order_items["shipping_limit_date"].dtypes}")
print(f"\n{reviews[date_cols_review].dtypes}")


<a id="2-handling-missing-values"></a>
### 2. Handling Missing Values
Missing values needed to be handled individually instead of being removed automatically.

**Orders dataset**

Review of missing delivery timestamps showed that most records correspond to orders with statuses such as "shipped", "processing", "canceled", or "unavailable". Therefore, missing delivery dates were treated as valid business scenarios rather than data errors.

A small number of inconsistent records were identified where orders were marked as "delivered" despite missing customer delivery timestamps.

The overview of the records with status "delivered" and missing values showed that most likely customers had not marked the order as received on their side, or that might have been a system bug.

Since those records represented less than 0.01% of the dataset and could negatively affect delivery time calculations, they were removed from the analysis by creating a delivery flag.

In [ ]:
# Checking statuses of the missing values
orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts()

In [ ]:
# Viewing records in status "delivered" but having missing values in "order_delivered_customer_date" column
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

In [ ]:
# Number of orders in status "delivered"
orders[orders["order_status"] == "delivered"]["order_status"].value_counts()

In [ ]:
# Delivery flag
orders["is_delivered"] = (
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna())
)
orders["is_delivered"].value_counts()

In [ ]:
# For our analysis goals we need only orders in "delivered" status, so creating new dataframe
orders_delivered = orders[orders["is_delivered"]].copy()

print(f"Orders total:    {len(orders):,}")
print(f"Orders delivered: {len(orders_delivered):,}")
print(f"Removed:         {len(orders) - len(orders_delivered):,}")

**Products dataset**

Analysis showed that products with missing metadata were still actively used in customer transactions and generated revenue.

A total of 1,603 order items were linked to products with missing category and description information. Therefore, removing these rows would lead to revenue loss and biased business analysis, so the best option was to replace missing product categories  with 'unknown' value and missing numerical data with median values.


In [ ]:
# Overviewing products w/o category 
products[products["product_category_name"].isnull()].head()

In [ ]:
# Checking if these products are getting ordered
missing_products = products[
    products["product_category_name"].isnull()
]

order_items[
    order_items["product_id"].isin(
        missing_products["product_id"]
    )
].shape

As we can see 1603 orders were made for the products with NaN values. So, we need to preserve these orders for accurate analysis. 

In [ ]:
# Filling missing values with "unknown"
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

# Replacing numerical characteristics with the median
numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]


for col in numeric_cols:
    products[col] = products[col].fillna(products[col].median())

print(products.isnull().sum())

**Reviews dataset**

Out of 99,224 reviews, only 11.7% contain a title and 41.3% contain a message.
This is an expected behavior for most of the customers. They simply leave a star rating without written feedback.

Since`review_comment_title` and `review_comment_message` columns will not be used here for further analysis there is no need of further cleaning manipulations with this data. Only `review_score` will be used.


In [ ]:
# How many reviews in total have some text?
total = len(reviews)
has_title = reviews["review_comment_title"].notna().sum()
has_message = reviews["review_comment_message"].notna().sum()

print(f"Total reviews:      {total:,}")
print(f"With title:         {has_title:,} ({has_title/total*100:.1f}%)")
print(f"With message:       {has_message:,} ({has_message/total*100:.1f}%)")

<a id="3-feature-engineering"></a>
### 3. Feature Engineering

To support further business analysis, several new analytical features were created across the datasets.

**Orders Dataset:** 

The following delivery and time-related features were added:

- `delivery_delay_days` — number of days between purchase and actual delivery
- `is_late` — boolean flag: True if delivered after the estimated delivery date
- `purchase_year_month` — year and month of purchase for time series analysis
- `purchase_month` / `purchase_year` — extracted from the purchase timestamp to analyze seasonality and yearly trends

These features support logistics performance analysis, customer satisfaction evaluation, and revenue trend exploration.

**Payments Dataset:**

Payment information was aggregated to the order level:

- `payments_agg` — payments aggregated to one row per order (total value)

This step was necessary because some orders contain multiple payment transactions.

**Reviews Dataset:**

A sentiment grouping feature was created based on review scores:

- `review_sentiment_group`
  - Positive → review scores 4–5
  - Neutral → review score 3
  - Negative → review scores 1–2

This feature simplifies customer satisfaction analysis and allows easier segmentation of review behavior.

In [ ]:
# Features for revenue over time and seasonality analysis
orders_delivered["purchase_year_month"] = (
    orders_delivered["order_purchase_timestamp"].dt.to_period("M").astype(str)
)
orders_delivered["purchase_month"] = orders_delivered["order_purchase_timestamp"].dt.month
orders_delivered["purchase_year"] = orders_delivered["order_purchase_timestamp"].dt.year


In [ ]:
# Delivery delay vs review score
orders_delivered["delivery_delay_days"] = (
    orders_delivered["order_delivered_customer_date"]
    - orders_delivered["order_purchase_timestamp"]
).dt.days

orders_delivered["is_late"] = (
    orders_delivered["order_delivered_customer_date"].dt.date
    > orders_delivered["order_estimated_delivery_date"].dt.date
)

In [ ]:
orders_delivered.head(3)

In [ ]:
# Payments aggregation
payments_agg = (
    payments.groupby("order_id")
    .agg(
        total_order_value=("payment_value", "sum"),
        payment_type=("payment_type", "first"),
        installments=("payment_installments", "max")
    )
    .reset_index()
)

payments_agg.head(3)

In [ ]:
# Review sentiment 
reviews["review_sentiment_group"] = np.where(
    reviews["review_score"] >= 4, "Positive",
    np.where(reviews["review_score"] == 3, "Neutral", "Negative")
)

reviews.head(3)

<a id="step-iv-creating-master-table"></a>
## Step IV. Creating Master Table

A master table `master_df` was created by joining all relevant tables using `order_items` as the foundation because it has the most detailed data — one row for every item in an order.

**Joins applied:**
- `orders_delivered` — order-level info: dates, delivery delay, late flag
- `payments_agg` — total order value aggregated from payments
- `customers` — customer state
- `products_en` — product category in English
- `reviews` — review score and sentiment group (left join — not all orders have a review)
- `sellers` — seller state

**Result:** 110,186 rows × 22 columns

**Remaining nulls after cleaning:**
- `review_score` / `review_sentiment_group` — 827 nulls, expected as not all orders have a review
- `product_category_name_english` — filled with `"unknown"` for products without a category

In [ ]:
# Merging products with English categories
products_en = products.merge(categories, on="product_category_name", how="left")
print(f"products_en: {len(products_en):,}")

In [ ]:
products_en.head(3)

In [ ]:
# Merging tabels into a Master table
master_df = (
    order_items
    .merge(orders_delivered[["order_id", "customer_id", 
                              "order_purchase_timestamp",
                              "purchase_year_month", "purchase_month", 
                              "purchase_year", "delivery_delay_days", 
                              "is_late"]], on="order_id")
    .merge(payments_agg, on="order_id")
    .merge(customers[["customer_id", "customer_state"]], on="customer_id")
    .merge(products_en[["product_id", "product_category_name_english"]], on="product_id")
    .merge(reviews[["order_id", "review_score", "review_sentiment_group"]]
           .drop_duplicates("order_id"), on="order_id", how="left")
    .merge(sellers[["seller_id", "seller_state"]], on="seller_id")
)

print(f"Master table shape: {master_df.shape}")
master_df.head()

In [ ]:
master_df.isnull().sum()[master_df.isnull().sum() > 0]

In [ ]:
# Checking which product_ids don't have a category in English
master_df[master_df["product_category_name_english"].isnull()][["product_id", "product_category_name_english"]].head()

In [ ]:
# Checking original category name for missing values
master_df[master_df["product_category_name_english"].isnull()][["product_id"]].merge(
    products_en[["product_id", "product_category_name", "product_category_name_english"]], 
    on="product_id"
).head(10)

In [ ]:
# Filling NaN values in product_category_name_english column with 'unknown' values
master_df["product_category_name_english"] = (
    master_df["product_category_name_english"].fillna("unknown")
)

master_df.isnull().sum()[master_df.isnull().sum() > 0]

**`repeat_customers` table**

A separate aggregation was created to analyze repeat purchase behavior. `repeat_customers` was built from `orders_delivered` at the customer level — one row per customer with the total number of unique orders.

A customer is considered a repeat buyer if they placed more than one order (`is_repeat_customer = True`).

This table was intentionally kept separate from `master_df` as it operates at the customer level, while the master table is at the order-item level.

In [ ]:
# DataFrame for repeat purchase behavior analysis
repeat_customers = (
    orders_delivered
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
    .groupby("customer_unique_id")["order_id"]
    .nunique()
    .reset_index()
    .rename(columns={"order_id": "order_count"})
)
repeat_customers["is_repeat_customer"] = repeat_customers["order_count"] > 1

# Checking if it was created correctly
repeat_customers.describe()


In [ ]:
# Data quality check — orders without payment record
order_id = "bfbd0f9bdef84302105ad712db648a6c"

orders_without_payment = orders_delivered[
    ~orders_delivered["order_id"].isin(payments["order_id"])
]
print(f"Orders without payment: {len(orders_without_payment)}")

**Note:** 1 delivered order was excluded from the Master Table due to missing payment record. This is a data quality issue in the source dataset and has no material impact on the analysis.

<a id="step-v-data-analysis"></a>
## Step V. Data Analysis

Before performing the analysis, incomplete historical periods were identified and excluded to ensure consistency and reliability of business metrics.

Excluded periods:
- `2016-10` — partial first month in dataset (265 orders)
- `2016-12` — only 1 recorded order (likely missing historical data)

All subsequent analyses are based on the validated period from January 2017 to August 2018.

In [ ]:
# Filter valid analysis period
clean_df = master_df[
    (master_df["order_purchase_timestamp"] >= "2017-01-01") &
    (master_df["order_purchase_timestamp"] <= "2018-08-31")
].copy()

print("Analysis period:")
print(
    clean_df["order_purchase_timestamp"].min(),
    "→",
    clean_df["order_purchase_timestamp"].max()
)

print(f"\nMaster tabel dataset shape: {master_df.shape}")
print(f"\nFiltered dataset shape: {clean_df.shape}")

<a id="executive-business-overview"></a>
## Executive Business Overview

This section provides an overview of the ecommerce platform’s overall business performance and operational health.

Key performance indicators (KPIs) were calculated to evaluate:

- revenue generation
- transaction volume
- customer satisfaction
- delivery performance
- customer retention
- revenue concentration

These metrics establish the foundation for the deeper analytical sections explored later in the project.

<a id="performance-kpis"></a>
### Performance KPIs

The following KPIs summarize the platform’s overall commercial performance during the validated analysis period from January 2017 to August 2018.

In [ ]:
total_revenue = clean_df["price"].sum()
total_orders = clean_df["order_id"].nunique()
avg_order_value = clean_df.groupby("order_id")["price"].sum().mean()
avg_review_score = clean_df["review_score"].mean()
late_delivery_rate = clean_df["is_late"].mean() * 100
repeat_rate = repeat_customers["is_repeat_customer"].mean() * 100
clv = total_revenue / customers["customer_unique_id"].nunique()
rev_per_seller = total_revenue / clean_df["seller_id"].nunique()


print(f"Total Revenue:          {total_revenue:,.2f}")
print(f"Total Orders:           {total_orders:,}")
print(f"Avg Order Value:        {avg_order_value:.2f}")
print(f"Avg Review Score:       {avg_review_score:.2f} / 5.0")
print(f"Late Delivery Rate:     {late_delivery_rate:.1f}%")
print(f"Repeat Purchase Rate:   {repeat_rate:.1f}%")
print(f"Avg Customer LTV:       {clv:.2f}")
print(f"Avg Revenue per Seller: {rev_per_seller:.2f}")

**Observations**
- The ecommerce platform generated more than R$13.1M in revenue across over 96k completed orders, indicating strong marketplace activity.

- The average customer review score remains relatively high (4.08 / 5.0), suggesting generally positive customer experiences.

- The late delivery rate is relatively low (6.6%), indicating stable logistics performance overall. However, delayed deliveries still show a negative impact on customer satisfaction in subsequent analyses.

- The average order value and customer lifetime value (R$137.15) suggest moderate customer spending behavior across the platform.

- Repeat purchasing behavior remains limited, with a repeat purchase rate of only 3%. This indicates that the business currently depends more heavily on customer acquisition than long-term retention.

The KPI analysis suggests that the platform operates with healthy sales volume and generally positive customer satisfaction, but faces important retention challenges. Improving customer loyalty and repeat purchase behavior may represent one of the largest opportunities for long-term growth.

<a id="revenue-and-sales-performance"></a>
## Revenue and Sales Performance

This section explores marketplace growth, revenue dynamics, sales concentration, seller performance, and customer purchasing behavior across product categories, regions, and payment methods.

<a id="revenue-and-order-volume-over-time"></a>
### Revenue and Order Volume Over Time

To evaluate overall business growth and seasonality patterns, monthly revenue and order volume trends were analyzed over time.

This analysis helps identify:

- long-term revenue growth trends
- changes in customer demand
- seasonal purchasing behavior
- periods of accelerated or weakened marketplace activity

In [ ]:
# Revenue and order volume over time
monthly_stats = (
    clean_df.groupby("purchase_year_month")
    .agg(
        total_revenue=("price", "sum"),
        orders_count=("order_id", "nunique")
    )
    .reset_index()
    .sort_values("purchase_year_month")
)

monthly_stats.head(10)

The visualization below compares monthly revenue generation and order activity using a dual-axis chart.

In [ ]:
# Visualization dual axis: revenue + order volume
fig, ax1 = plt.subplots(figsize=(12, 5))

ax2 = ax1.twinx()

ax1.bar(
    monthly_stats["purchase_year_month"],
    monthly_stats["total_revenue"],
    color="#5BA4CF",
    alpha=0.8,
    label="Revenue"
)

ax2.plot(
    monthly_stats["purchase_year_month"],
    monthly_stats["orders_count"],
    color="#E53935",
    linewidth=2,
    marker="o",
    markersize=4,
    label="Orders"
)

ax1.set_title("Monthly revenue and order volume (2017–2018)", fontsize=14)
ax1.set_xlabel("Month")
ax1.set_ylabel("Revenue (BRL)")
ax2.set_ylabel("Number of orders")
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax1.set_xticks(range(len(monthly_stats)))
ax1.set_xticklabels(monthly_stats["purchase_year_month"], rotation=45, ha="right")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.savefig("viz_1_monthly_revenue.png", dpi=150)
plt.show()

We can notice on the graph an overall growing trend and a strong positive relationship between monthly revenue and order volume, indicating that business growth was primarily driven by increasing customer activity. Both metrics move together throughout the period, suggesting the average order value remained stable over time.

From early 2017 to late 2017, the platform experienced rapid growth in both sales and customer demand.

Several important seasonal patterns can be observed:

- A major sales peak occurred in November 2017, when both revenue and order volume reached their highest levels. This spike was likely influenced by Black Friday and holiday shopping campaigns.
- A noticeable decline followed in December 2017, which may indicate post-promotion normalization after the November sales peak.
- During 2018, revenue remained relatively stable at a high level, suggesting that the platform maintained strong customer demand.

The analysis suggests that the platform experienced strong marketplace growth during the observed period. Seasonal spikes highlight the importance of promotional campaigns and operational planning during high-demand periods.

<a id="revenue-contribution-by-product-category"></a>
### Revenue Contribution by Product Category

This analysis examines which product categories generate the highest revenue and how sales are distributed across the platform’s product portfolio.

Understanding highest revenue category helps to identify:

- the platform’s strongest-performing product segments
- customer purchasing preferences
- dependency on specific product categories
- opportunities for marketing focus

In [ ]:
# Revenue by product category
category_revenue = (
    clean_df.groupby("product_category_name_english")
    .agg(
        total_revenue=("price", "sum"),
        orders_count=("order_id", "nunique")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

# Revenue share %
category_revenue["revenue_share_pct"] = (
    category_revenue["total_revenue"] / category_revenue["total_revenue"].sum() * 100
).round(2)

category_revenue.head(10)

In [ ]:
# How many categories in total? 
print(f'Total number of product categories: {clean_df["product_category_name_english"].nunique()}')

In [ ]:
# Viz of top 10 product categories
fig, ax = plt.subplots(figsize=(10, 6))

top10 = category_revenue.head(10)

ax.barh(
    top10["product_category_name_english"],
    top10["total_revenue"],
    color=sns.color_palette("Blues_r", 10)
)

ax.set_title("Top 10 product categories by revenue", fontsize=14)
ax.set_xlabel("Total revenue (BRL)")
ax.set_ylabel("Category")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))


for i, (rev, pct) in enumerate(zip(top10["total_revenue"], top10["revenue_share_pct"])):
    ax.text(rev + 5000, i, f"{pct}%", va="center", fontsize=9)

ax.invert_yaxis()
ax.xaxis.grid(True, linestyle="--", alpha=0.5)
ax.yaxis.grid(False)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("viz_2_category_revenue.png", dpi=150)
plt.show()

The highest revenue-generating category was Health & Beauty, contributing approximately 9.3% of total sales revenue, followed by:
- Watches & Gifts 
- Bed & Bath Table 
- Sports & Leisure 
- Computers & Accessories

The results suggest that customers show particularly strong demand for: personal care products, home-related items, electronics and accessories, lifestyle and leisure products.

**Key observations:**
- Revenue is spread across many categories — the top category holds less than 10% of total sales, indicating a well-diversified product portfolio
- `watches_gifts` has fewer orders than `bed_bath_table` (5,493 vs 9,272) but higher revenue, suggesting higher average order value
- Top 10 categories account for ~62% of total revenue

From a business perspective, these categories may represent the most valuable areas for:
- marketing investments,
- inventory prioritization,
- logistics optimization,
- seller partnerships.

<a id="revenue-contribution-by-state"></a>
### Revenue Contribution by State

A pivot table was created to summarize key metrics by customer state: total revenue, number of orders, unique customers, and revenue share.

This analysis examines how revenue and customer activity are distributed across Brazilian states.

The goal is to identify:

- the platform’s strongest regional markets
- geographic revenue concentration
- customer distribution patterns
- potential opportunities for regional expansion

In [ ]:
# Pivot table — state performance summary
pivot_table = clean_df.merge(
    customers[["customer_id", "customer_unique_id"]], on="customer_id"
).pivot_table(
    index="customer_state",
    values=["price", "order_id", "customer_unique_id"],
    aggfunc={
        "price": "sum",
        "order_id": "nunique",
        "customer_unique_id": "nunique"
    }
).round(2)

pivot_table.columns = ["unique_customers", "orders_count", "total_revenue"]
pivot_table["revenue_share_pct"] = (
    pivot_table["total_revenue"] / pivot_table["total_revenue"].sum() * 100
).round(2)

pivot_table = pivot_table.sort_values("total_revenue", ascending=False)
pivot_table.head(10)

The analysis shows that revenue and customer activity are heavily concentrated in a small number of Brazilian states.

São Paulo (SP) is the platform’s dominant market, contributing approximately 38% of total revenue and the largest customer base.

Rio de Janeiro (RJ) and Minas Gerais (MG) are the next strongest-performing states, together contributing a significant share of marketplace revenue and orders - 25% of total revenue. 

Southern and southeastern states generally demonstrate stronger commercial performance compared to other regions.

In most states, the number of orders slightly exceeds the number of unique customers, indicating limited repeat purchasing behavior across the platform.

The results suggest that marketplace activity is strongly concentrated in economically developed regions with larger populations and stronger purchasing power.

While the platform generates strong revenue in its main markets, it also depends heavily on a small number of regions. Expanding into less active states could help create more balanced long-term growth.

<a id="seller-performance-and-customer-satisfaction"></a>
### Seller Performance and Customer Satisfaction

This analysis evaluates seller performance by comparing revenue generation, order volume, and average customer review scores.

The goal is to identify:

- top-performing sellers
- relationships between sales volume and customer satisfaction
- sellers with strong operational performance
- potential risks among high-volume but low-rated merchants

In [ ]:
# Top sellers by revenue and satisfaction
seller_stats = (
    clean_df.groupby(["seller_id", "seller_state"])
    .agg(
        total_revenue=("price", "sum"),
        orders_count=("order_id", "nunique"),
        avg_review_score=("review_score", "mean")
    )
    .round(2)
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

seller_stats.head(10)

The highest-performing seller generated more than R$226k in revenue while maintaining a strong average review score above 4.1. Several other top sellers also combined high sales volume with consistently positive customer ratings.

At the same time, some high-revenue sellers received noticeably lower review scores, suggesting that strong sales performance does not always guarantee positive customer experience.

The platform may benefit from monitoring high-volume sellers with lower customer ratings and identifying operational best practices among top-performing sellers.

In [ ]:
# Scatter plot: revenue vs review score
fig, ax = plt.subplots(figsize=(10, 6))

# All sellers as background
ax.scatter(
    seller_stats["total_revenue"],
    seller_stats["avg_review_score"],
    alpha=0.3,
    s=20,
    color="#90CAF9"
)

# Top 10 highlighted
top10_sellers = seller_stats.head(10)
ax.scatter(
    top10_sellers["total_revenue"],
    top10_sellers["avg_review_score"],
    alpha=0.9,
    s=80,
    color="#1565C0",
    edgecolors="white",
    linewidth=0.8,
    label="Top 10 sellers"
)

ax.set_title("Seller revenue vs crustomer satisfaction", fontsize=14)
ax.set_xlabel("Total revenue (BRL)")
ax.set_ylabel("Average review score (1–5)")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
ax.legend()
ax.grid(False)

plt.tight_layout()
plt.savefig("viz_3_seller_stats.png", dpi=150)
plt.show()

The scatterplot highlights that most sellers are concentrated in the lower revenue range, while only a small number of merchants generate exceptionally high revenue. The visualization suggests that marketplace revenue is heavily concentrated among a limited group of top-performing sellers.

The relationship between revenue and customer satisfaction is mixed: some high-revenue sellers maintain strong review scores, while others receive noticeably lower ratings despite large sales volumes.

The results suggest that seller growth should be evaluated not only by revenue performance, but also by customer satisfaction and operational quality.

<a id="payment-method-analysis"></a>
### Payment Method Analysis

This analysis explores customer payment preferences and compares transaction volume and average order value across different payment methods.

The goal is to understand:

- the most commonly used payment types
- differences in customer purchasing behavior
- how payment methods relate to average order value
- potential opportunities to improve checkout experience and payment flexibility

In [ ]:
# Payment methods analysis
payment_stats = (
    payments.groupby("payment_type")
    .agg(
        transactions=("order_id", "count"),
        avg_order_value=("payment_value", "mean"),
        total_value=("payment_value", "sum")
    )
    .round(2)
    .reset_index()
    .sort_values("transactions", ascending=False)
)

payment_stats["transactions_pct"] = (
    payment_stats["transactions"] / payment_stats["transactions"].sum() * 100
).round(2)

payment_stats

In [ ]:
# Removing not_defined 3 transactions with zero value
payment_clean = payment_stats[payment_stats["payment_type"] != "not_defined"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ["#1565C0", "#42A5F5", "#90CAF9", "#BBDEFB"]

# Donut chart: transaction share
sizes = payment_clean["transactions_pct"]
labels = payment_clean["payment_type"]

wedges, texts, autotexts = axes[0].pie(
    sizes,
    labels=None, 
    colors=colors,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1, "width": 0.6},
    pctdistance=0.75 
)
for at in autotexts:
    at.set_fontsize(10)

axes[0].legend(
    wedges, labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
    fontsize=9,
    frameon=False
)
axes[0].set_title("Transaction share by payment type", fontsize=13)

# Horizontal bar: avg order value
axes[1].barh(payment_clean["payment_type"], payment_clean["avg_order_value"], color=colors)
axes[1].set_title("Average order value by payment type", fontsize=13)
axes[1].set_xlabel("Average order value (BRL)")
axes[1].invert_yaxis()
axes[1].grid(False)
for i, v in enumerate(payment_clean["avg_order_value"]):
    axes[1].text(v + 1, i, f"R$ {v:.0f}", va="center", fontsize=9)

plt.suptitle("Payment methods analysis", fontsize=14)
plt.tight_layout()
plt.savefig("viz_4_payment_methods.png", dpi=150, bbox_inches="tight")
plt.show()

Credit cards are the dominant payment method on the platform, accounting for nearly 74% of all transactions.

Boleto was the second most popular payment method, representing approximately 19% of purchases, while voucher and debit card payments have relatively small shares of total transactions.

Payment methods also differ in average order value:
  - credit card purchases show the highest average order value,
  - boleto and debit card transactions remain relatively high,
  - voucher payments have substantially lower average purchase amounts.

The results suggest that customers using credit cards tend to place larger purchases compared to customers using other payment methods.

The strong dominance of credit card transactions suggests that payment flexibility likely plays an important role in customer purchasing behavior.

The platform may benefit from optimizing the credit card checkout experience and supporting installment payment options to improve conversion and transaction value.

<a id="customer-experience-and-logistics-performance"></a>
## Customer Experience and Logistics Performance

This section analyzes operational performance and customer satisfaction, focusing on delivery efficiency, review scores, regional logistics differences, and factors contributing to customer dissatisfaction.

<a id="impact-of-delivery-delays-on-customer-satisfaction"></a>
### Impact of Delivery Delays on Customer Satisfaction

This analysis examines how delivery performance affects customer review scores.

The goal is to evaluate whether late deliveries lead to lower customer satisfaction and to better understand the operational impact of logistics performance on the overall marketplace experience.

In [ ]:
# Delivery delay vs review score
delay_vs_review = (
    clean_df[["delivery_delay_days", "review_score", "is_late"]]
    .dropna()
)

# Average review score: late vs on time
delay_summary = (
    delay_vs_review.groupby("is_late")["review_score"]
    .agg(["mean", "median", "count"])
    .round(2)
)
delay_summary.index = ["On time", "Late"]
delay_summary

A strong relationship exists between delivery performance and customer satisfaction.

Orders delivered on time received an average review score of 4.21, while late deliveries received a much lower average score of 2.26.

Median review scores highlight this difference even more clearly:
  - on-time deliveries most commonly received the maximum rating of 5 stars,
  - late deliveries most frequently received ratings close to 1 star.

The results show that delivery performance is one of the most important operational factors affecting customer satisfaction and marketplace reputation.

In [ ]:
# Box-plot
fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=delay_vs_review,
    x="is_late",
    y="review_score",
    hue="is_late",
    palette={False: "#1565C0", True: "#F5A623"},
    legend=False,
    showmeans=True,
    meanprops={
        "linestyle": "--",
        "linewidth": 1,
        "color": "black"
    },
    meanline=True,
    ax=ax
)

ax.set_title("Review scores: on-time vs late deliveries", fontsize=14)
ax.set_xlabel("Delivery status")
ax.set_ylabel("Review score (1–5)")
ax.set_xticks([0, 1])
ax.set_xticklabels(["On time", "Late"])
ax.grid(False)

plt.tight_layout()
plt.savefig("viz_5_review_vs_delay.png", dpi=150)
plt.show()

In [ ]:
# Sentiment distribution by delivery status
sentiment_delivery = (
    clean_df.groupby(["is_late", "review_sentiment_group"])
    .size()
    .reset_index(name="count")
)
sentiment_delivery["pct"] = (
    sentiment_delivery.groupby("is_late")["count"]
    .transform(lambda x: x / x.sum() * 100)
)

pivot = sentiment_delivery.pivot(
    index="is_late", columns="review_sentiment_group", values="pct"
)[["Positive", "Neutral", "Negative"]]

fig, ax = plt.subplots(figsize=(8, 5))
pivot.plot(
    kind="bar", stacked=True, ax=ax,
    color=["#1565C0", "#90CAF9", "#F5A623"],
    width=0.5
)
ax.set_title("Sentiment distribution: on-time vs late deliveries", fontsize=14)
ax.set_xlabel("")
ax.set_xticklabels(["On-time", "Late"], rotation=0)
ax.set_ylabel("Share of reviews (%)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(title="Sentiment", bbox_to_anchor=(1, 1))
ax.grid(False)
plt.tight_layout()
plt.savefig("viz_6_sentiment_delivery.png", dpi=150)
plt.show()

Orders delivered on time show consistently high review scores, with most ratings concentrated between 4 and 5 stars.

Late deliveries demonstrate significantly lower review scores and much greater variability in customer feedback.

The median review score for delayed orders is close to 1 star, indicating a strong negative customer reaction to delivery issues.

The visualization confirms that logistics performance is a critical driver of customer satisfaction and overall customer experience.

<a id="hypothesis-test-late-deliveries"></a>
### Hypothesis Test: Late Deliveries and Review Scores 

A statistical hypothesis test was performed to determine whether late deliveries lead to lower customer review scores.

H₀: there is no significant difference in review scores between on-time and late deliveries.

H₁: late deliveries have significantly lower review scores.


In [ ]:
# Preparing samples
on_time_scores = clean_df[clean_df["is_late"] == False]["review_score"].dropna()
late_scores    = clean_df[clean_df["is_late"] == True]["review_score"].dropna()

# Performing t-test
t_stat, p_value = stats.ttest_ind(on_time_scores, late_scores, equal_var=False)

print(f"On time — mean score: {on_time_scores.mean():.2f}, n = {len(on_time_scores):,}")
print(f"Late    — mean score: {late_scores.mean():.2f}, n = {len(late_scores):,}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value:     {p_value:.10f}")

if p_value < 0.05:
    print("\nReject H0: late deliveries significantly lower review scores.")
else:
    print("\nFail to reject H0.")

A two-sample t-test was used to compare review scores between on-time and late deliveries.

The average review score for on-time deliveries was 4.21, while late deliveries received an average score of 2.26.

Since the p-value is significantly below 0.05, the null hypothesis was rejected.

The results confirm that late deliveries are associated with significantly lower customer review scores.

<a id="delivery-performance-and-customer-satisfaction-by-state"></a>
### Delivery Performance and Customer Satisfaction by State

This analysis explores how delivery performance varies across Brazilian states and whether longer delivery times are associated with lower customer review scores.

The goal is to identify regional logistics patterns and evaluate how operational efficiency impacts customer satisfaction.

In [ ]:
# Average delivery time and review score by state
state_stats = (
    clean_df.groupby("customer_state")
    .agg(
        avg_delivery_days=("delivery_delay_days", "mean"),
        avg_review_score=("review_score", "mean"),
        orders_count=("order_id", "nunique")
    )
    .round(2)
    .reset_index()
    .sort_values("avg_delivery_days", ascending=False)
)

state_stats.head(10)

In [ ]:
# Correlation between delivery time and review score
correlation = state_stats["avg_delivery_days"].corr(state_stats["avg_review_score"])
print(f"Pearson correlation: {correlation:.3f}")

# Scatter plot: delivery days vs review score
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    state_stats["avg_delivery_days"],
    state_stats["avg_review_score"],
    s=state_stats["orders_count"] / 10,
    alpha=0.7,
    color="#90CAF9",
    edgecolors="#1565C0",
    linewidth=0.8
)

# State labels
for _, row in state_stats.iterrows():
    ax.annotate(
        row["customer_state"],
        (row["avg_delivery_days"], row["avg_review_score"]),
        fontsize=8,
        ha="left",
        xytext=(4, 4),
        textcoords="offset points"
    )

# Trend line
m, b = np.polyfit(state_stats["avg_delivery_days"], state_stats["avg_review_score"], 1)
x_range = np.linspace(state_stats["avg_delivery_days"].min(), 
                       state_stats["avg_delivery_days"].max(), 100)
ax.plot(x_range, m * x_range + b, color="#E53935", linewidth=2, 
        linestyle="--", label=f"Trend (r={correlation:.2f})")

ax.set_title("Average delivery time vs customer satisfaction by state", fontsize=14)
ax.set_xlabel("Average delivery time (days)")
ax.set_ylabel("Average review score (1–5)")
ax.legend()
ax.grid(False)

plt.tight_layout()
plt.savefig("viz_7_state_delivery_satisfaction.png", dpi=150)
plt.show()

A moderate negative correlation exists between average delivery time and customer satisfaction across Brazilian states (Pearson correlation = -0.31).

In general, states with longer delivery times tend to receive lower average review scores, suggesting that slower logistics negatively affect customer experience.

However, the correlation is not very strong, indicating that delivery time is only one of several factors influencing customer satisfaction.

- São Paulo (SP), which generated the highest order volume, demonstrated one of the shortest delivery times while maintaining high customer review scores. This may reflect stronger logistics infrastructure and operational efficiency.

- Several geographically remote states, including RR, AP, and AM, experienced the longest average delivery times.

- States such as Maranhão (MA), Alagoas (AL), and Pará (PA) combined relatively long delivery times with lower customer review scores, indicating potential areas for logistics improvement.


The results reinforce the importance of logistics performance as a key driver of customer satisfaction and operational success. Improving delivery efficiency in remote and underperforming regions may help increase customer satisfaction and strengthen marketplace performance across different states.

<a id="low-rated-product-categories-and-customer-dissatisfaction"></a>
### Low-Rated Product Categories and Customer Dissatisfaction

This analysis identifies product categories with the lowest customer review scores and explores possible operational factors contributing to customer dissatisfaction.

The analysis considers:
- average review scores,
- delivery performance,
- late delivery rates,
- and category-specific operational characteristics.

In [ ]:
# Categories with lowest review scores
category_reviews = (
    clean_df.groupby("product_category_name_english")
    .agg(
        avg_review_score=("review_score", "mean"),
        total_reviews=("review_score", "count"),
        avg_delivery_days=("delivery_delay_days", "mean"),
        late_rate=("is_late", "mean")
    )
    .round(2)
    .reset_index()
)

# Min 100 reviews for statistical reliability
category_reviews = category_reviews[category_reviews["total_reviews"] >= 100]

category_reviews_sorted = category_reviews.sort_values("avg_review_score")
category_reviews_sorted.head(10)

In [ ]:
# Bottom 10 categories by review score
bottom10 = category_reviews_sorted.head(10)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    bottom10["product_category_name_english"],
    bottom10["avg_review_score"],
    color=sns.color_palette("Oranges", 10)
)

ax.set_title("Bottom 10 categories by average review score", fontsize=14)
ax.set_xlabel("Average review score (1–5)")
ax.set_ylabel("Category")
ax.set_xlim(0, 5)
ax.invert_yaxis()
ax.grid(False)

# Add score labels
for i, (score, late) in enumerate(zip(bottom10["avg_review_score"],
                                       bottom10["late_rate"])):
    ax.text(score + 0.05, i, f"{score} (late: {late*100:.0f}%)",
            va="center", fontsize=9)

plt.tight_layout()
plt.savefig("viz_8_category_reviews.png", dpi=150)
plt.show()

Office Furniture received the lowest average review score (3.52) among categories with sufficient review volume. This category also showed one of the longest average delivery times (over 20 days), suggesting that logistics challenges may strongly contribute to customer dissatisfaction.

Other relatively low-rated categories included:
  - Fixed Telephony,
  - Fashion Male Clothing,
  - Audio,
  - Home Comfort,
  - and several furniture-related categories.

Furniture categories generally combined longer delivery times with lower review scores, indicating that shipping complexity and delivery performance may negatively affect customer experience.

For electronics and clothing categories, dissatisfaction may also be influenced by product quality, unmet expectations, or product fit issues rather than delivery performance alone.

The results suggest several opportunities to improve customer satisfaction:

- optimize shipping and delivery processes,
- improve packaging quality for large or fragile products,
- strengthen seller quality control,
- and provide clearer delivery communication and customer support.

<a id="customer-retention-and-segmentation"></a>
## Customer Retention and Segmentation

This section evaluates customer retention, repeat purchase behavior, and customer segmentation using RFM and cohort analysis to better understand long-term customer engagement.

<a id="repeat-purchase-behavior"></a>
### Repeat Purchase Behavior

This analysis evaluates customer retention by measuring how many customers returned to place additional orders.

The goal is to understand repeat purchase behavior and assess the marketplace’s ability to retain customers over time.

In [ ]:
# Repeat purchase rate
print(f"Total unique customers:   {len(repeat_customers):,}")
print(f"Repeat customers:         {repeat_customers['is_repeat_customer'].sum():,}")
print(f"Repeat purchase rate:     {repeat_customers['is_repeat_customer'].mean()*100:.2f}%")

# Distribution of order counts
repeat_customers["order_count"].value_counts().sort_index().head(10)

**Note:** Analysis is based on 93,350 customers with at least one delivered order, excluding customers whose orders were cancelled or not yet delivered.

In [ ]:
# Order count distribution
fig, ax = plt.subplots(figsize=(8, 5))

order_dist = repeat_customers["order_count"].value_counts().sort_index().head(6)

ax.bar(
    order_dist.index.astype(str),
    order_dist.values,
    color=sns.color_palette("Blues_r", 6)
)

ax.set_title("Distribution of orders per customer", fontsize=14)
ax.set_xlabel("Number of orders")
ax.set_ylabel("Number of customers")
ax.grid(False)

# Value labels on bars
for i, v in enumerate(order_dist.values):
    ax.text(i, v + 100, f"{v:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("viz_9_repeat_customers.png", dpi=150)
plt.show()

Out of more than 93,000 unique customers, only approximately 3% returned to place additional orders. Most repeat customers made only two purchases, while customers with three or more orders were extremely rare. The results indicate very low customer retention during the observed period.

The marketplace appears to rely heavily on acquiring new customers rather than retaining existing ones.

Improving repeat purchase behavior may represent a major opportunity for long-term revenue growth and customer lifetime value expansion.

Potential retention strategies may include:
- loyalty programs,
- personalized product recommendations,
- post-purchase engagement,
- and targeted remarketing campaigns.

<a id="customer-segmentation-analysis-rfm"></a>
### Customer Segmentation Analysis (RFM)

To better understand customer behavior and purchasing patterns, an RFM (Recency, Frequency, Monetary) analysis was performed.

RFM segmentation groups customers based on:

- **Recency** — how recently a customer made a purchase
- **Frequency** — how often the customer places orders
- **Monetary Value** — how much revenue the customer generates

This approach helps identify high-value customers, low-engagement users, and potential retention risks.

Since the dataset shows relatively low repeat purchase behavior overall, the segmentation model was simplified into five business-oriented customer groups to improve interpretability and stakeholder readability.

In [ ]:
# Adding customer_unique_id from customers table
rfm_df = clean_df.merge(
    customers[["customer_id", "customer_unique_id"]], 
    on="customer_id",
    how="left"
)

print(f"Unique customer_id:        {rfm_df['customer_id'].nunique():,}")
print(f"Unique customer_unique_id: {rfm_df['customer_unique_id'].nunique():,}")

In [ ]:
# Snapshot date — day after last purchase
snapshot_date = rfm_df["order_purchase_timestamp"].max() + dt.timedelta(days=1)
print(f"Snapshot date: {snapshot_date.date()}")

# Calculate RFM metrics per unique customer
rfm = (
    rfm_df.groupby("customer_unique_id")
    .agg(
        recency=("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
        frequency=("order_id", "nunique"),
        monetary=("price", "sum")
    )
    .reset_index()
)

print(f"\nRFM table shape: {rfm.shape}")
print(f"\nRFM summary:")
rfm[["recency", "frequency", "monetary"]].describe().round(2)

In [ ]:
# RFM scores 1-5
rfm["r_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1])
rfm["f_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]
)
rfm["m_score"] = pd.qcut(rfm["monetary"], 5, labels=[1, 2, 3, 4, 5])

# Combined RFM score
rfm["rfm_score"] = (
    rfm["r_score"].astype(int)
    + rfm["f_score"].astype(int)
    + rfm["m_score"].astype(int)
)

print(f"RFM shape: {rfm.shape}")
print(f"\nScore distribution:")
rfm["rfm_score"].value_counts().sort_index()

In [ ]:
# Customer segmentation based on RFM scores
def label_segment(row):
    r = int(row["r_score"])
    f = int(row["f_score"])
    m = int(row["m_score"])

    if r >= 4 and f >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "New / Recent Customers"
    elif r <= 2 and f >= 3:
        return "At Risk"
    else:
        return "Low Engagement"

rfm["segment"] = rfm.apply(label_segment, axis=1)

# Segment summary
segment_summary = (
    rfm.groupby("segment")
    .agg(
        customers=("customer_unique_id", "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean"),
        total_revenue=("monetary", "sum")
    )
    .round(2)
    .sort_values("customers", ascending=False)
)

segment_summary["customers_pct"] = (
    segment_summary["customers"] / segment_summary["customers"].sum() * 100
).round(2)

segment_summary

The RFM analysis reveals several important patterns in customer behavior and retention dynamics within the ecommerce platform.

- The average purchase frequency is very low across all customer segments (~1 order per customer), indicating that most customers purchase only once.

- Nearly half of all customers belong to the **Low Engagement** or **At Risk** segments, suggesting weak customer retention and limited repeat purchasing behavior.

- Even the **Champions** segment demonstrates relatively low repeat activity (average frequency = 1.09), meaning that strong customer loyalty has not yet been fully established.

- A significant share of newly acquired customers fail to transition into loyal repeat buyers, highlighting opportunities for improving post-purchase engagement and retention strategies.

The analysis suggests that the business currently relies more heavily on customer acquisition than long-term customer retention. Improving customer loyalty programs, personalized marketing, and post-purchase engagement may help increase repeat purchase rates and long-term customer lifetime value.

**Recommendations**

- Improve customer retention initiatives through loyalty programs and personalized offers.
- Increase post-purchase engagement to encourage repeat purchases.
- Analyze factors driving one-time purchases and customer churn.

<a id="cohort-retention-analysis"></a>
### Cohort Retention Analysis

To better understand customer retention behavior over time, a cohort analysis was performed.

Customers were grouped into cohorts based on the month of their first purchase (`cohort_month`). Retention was then measured by tracking how many customers returned to place additional orders in subsequent months.

`cohort_index` represents the number of months since a customer's first purchase:
- `0` = acquisition month
- `1` = one month after first purchase
- `2` = two months later
- etc.
  
This analysis helps evaluate:

- customer retention trends
- repeat purchasing behavior
- long-term customer engagement
- sustainability of customer acquisition efforts
  
The retention matrix below shows the percentage of customers from each acquisition cohort who returned in future months.

In [ ]:
cohort_df = rfm_df.copy()

cohort_df["order_month"] = cohort_df["order_purchase_timestamp"].dt.to_period("M")
cohort_df["cohort_month"] = cohort_df.groupby("customer_unique_id")["order_month"].transform("min")

cohort_df["cohort_index"] = (
    (cohort_df["order_month"].dt.year - cohort_df["cohort_month"].dt.year) * 12 +
    (cohort_df["order_month"].dt.month - cohort_df["cohort_month"].dt.month)
)

cohort_data = (
    cohort_df.groupby(["cohort_month", "cohort_index"])["customer_unique_id"]
    .nunique()
    .reset_index()
)

cohort_pivot = cohort_data.pivot(
    index="cohort_month",
    columns="cohort_index",
    values="customer_unique_id"
)

cohort_retention = cohort_pivot.divide(cohort_pivot[0], axis=0) * 100
(
    cohort_retention.round(2)
    .style
    .background_gradient(cmap="Blues", axis=None, vmin=0, vmax=1)
    .format("{:.2f}%", na_rep="-")
    .set_caption("Cohort retention rate (%)")
)

**Key Findings**

- Customer retention drops sharply after the acquisition month across nearly all cohorts.

- Most cohorts retain only a very small percentage of customers after the first purchase, confirming weak repeat purchasing behavior.

- Retention patterns remain consistently low across both 2017 and 2018 cohorts, suggesting that low customer retention is a structural characteristic of the platform rather than a temporary anomaly.

- No significant improvement in retention is observed across newer customer cohorts, indicating that customer loyalty and long-term engagement remained relatively unchanged over time.

The business currently appears to operate with an acquisition-driven growth model, relying heavily on attracting new customers rather than maximizing long-term customer lifetime value.

Improving retention strategies — such as loyalty programs, personalized recommendations, post-purchase engagement, and remarketing campaigns — could significantly improve customer lifetime value and long-term profitability.

<a id="conclusions-and-business-recommendations"></a>
# Conclusions and Business Recommendations

## Revenue and Sales Performance

### Key Insights

- The platform showed strong revenue and order growth throughout 2017, with a major peak in November 2017 driven by Black Friday demand.
- By 2018, monthly revenue stabilized at approximately R$800K–980K, suggesting the business entered a more mature growth stage.
- Revenue and order volume moved closely together over time, indicating relatively stable average order values.
- The top 26% of total revenue was generated by the below categories:
  - `health_beauty`,
  - `watches_gifts`,
  - `bed_bath_table`,

- Revenue was heavily concentrated geographically:
  - São Paulo (SP) alone generated approximately 38% of total revenue,
  - the top 3 states (SP, RJ, MG) accounted for nearly 63%.
- Marketplace sales were also concentrated among a relatively small group of top-performing sellers.
- Credit cards dominated payment behavior, representing around 74% of all transactions and the highest average order value among payment methods.

### Business Recommendations

1. **Reduce geographic concentration risk**  
   Expanding marketing efforts in underrepresented states may diversify revenue sources and reduce dependence on a few core regions.

2. **Strengthen relationships with top-performing sellers**  
   Supporting high-performing merchants with operational tools and promotional opportunities may help sustain marketplace growth.

3. **Prepare strategically for seasonal demand peaks**  
   Black Friday and other high-demand periods require proactive inventory planning, logistics preparation, and seller coordination.

4. **Optimize the credit card checkout experience**  
   Since credit cards dominate transactions and generate the highest order values, improving payment flexibility and installment options could further increase conversion rates.


---

## Customer Experience and Logistics Performance

### Key Insights

- Delivery performance had a major impact on customer satisfaction.
- On-time deliveries achieved an average review score of 4.21, while late deliveries averaged only 2.26.
- Statistical testing confirmed that the difference in review scores between on-time and late deliveries was highly significant (p < 0.05).
- States with longer average delivery times generally showed lower customer satisfaction scores.
- Northern states such as RR, AP, and AM experienced the slowest delivery times.
- São Paulo combined short delivery times with high customer satisfaction, indicating stronger logistics efficiency.
- Product categories with the lowest ratings — especially `office_furniture` — were often associated with longer delivery times and elevated late-delivery rates.
- Large or fragile categories such as furniture appeared more sensitive to logistics and shipping quality issues.

### Business Recommendations

1. **Improve logistics performance in remote regions**  
   Partnering with regional logistics providers may reduce delivery delays and improve customer satisfaction in northern states.

2. **Prioritize late-delivery reduction initiatives**  
   Since delivery delays strongly impact customer reviews, improving shipping reliability should be treated as a core operational priority.

3. **Monitor low-rated high-volume sellers**  
   Sellers generating strong sales but poor customer ratings should be flagged for operational review and quality control.

4. **Improve shipping processes for furniture-related categories**  
   Better packaging, delivery coordination, and supplier management may help reduce dissatisfaction in bulky-product categories.

5. **Enhance customer communication during delivery**  
   More transparent tracking updates and delivery notifications could help reduce customer frustration during delays.


---

## Customer Retention and Segmentation

### Key Insights

- Customer retention was extremely low:
  - only around 3% of customers placed more than one order,
  - 97% were one-time buyers.
- Most repeat customers made only two purchases, while customers with three or more orders were very rare.
- RFM analysis showed that the customer base was dominated by `Low Engagement` and `At Risk` segments, which together represented nearly half of all customers.
- High-value segments such as `Champions` and `Loyal Customers` represented a smaller share of users but generated strong revenue contributions.
- `Champions` showed the highest average spending and purchase frequency, making them the most valuable customer segment.
- Cohort retention analysis revealed a sharp drop in retention immediately after the first purchase month across nearly all cohorts.
- Retention patterns remained consistently weak throughout both 2017 and 2018, indicating that low repeat purchasing was a structural characteristic of the business rather than a temporary issue.
- The marketplace appeared to rely primarily on continuous customer acquisition rather than long-term customer retention.

### Business Recommendations

1. **Develop customer retention strategies**  
   Loyalty programs, personalized recommendations, remarketing campaigns, and post-purchase engagement initiatives could help improve repeat purchase behavior.

2. **Focus on high-value customer segments**  
   Retaining `Champions` and `Loyal Customers` may generate higher long-term value than relying mainly on acquiring new users.

3. **Increase post-purchase engagement**  
   Email campaigns, personalized offers, and targeted promotions could encourage second and third purchases.

4. **Investigate drivers of one-time purchasing behavior**  
   The extremely low retention rate suggests possible issues related to customer experience, pricing competitiveness, product quality, or marketplace differentiation.

5. **Track retention KPIs continuously**  
   Monitoring cohort retention and RFM segment movement over time may help measure the effectiveness of future retention initiatives.

<a id="tableau-dashboard"></a>
# Tableau Dashboard

Interactive dashboard published on Tableau Public:
[Brazilian E-Commerce Analysis](https://public.tableau.com/views/BrazilianE-CommerceAnalysis_17790304439800/SalesOperations?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

**Dashboard 1 — Sales & Operations Overview:** revenue trend, geographic distribution, top categories, payment methods

**Dashboard 2 — Customer Experience:** delivery performance by state, lowest rated categories

**Dashboard 3 — Customer Retention & Segmentation:** segment distribution, RFM segment analysis, retention heatmap
